<a href="https://colab.research.google.com/github/AngelTroncoso/Feature-Engineering/blob/main/actividad8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformando Datos: Feature Engineering
## Eliminación Recursiva de Características (RFE)

¡Hola! Qué alegría que sigas con nosotros en esta ruta de **Transformando Datos: Feature Engineering**. Hoy vamos a dominar una técnica que separa a los analistas promedio de los expertos: la **Elimación Recursiva de Características**. Prepárate porque esto transformará tu forma de trabajar con modelos complejos.

### ¿Por qué RFE?
A veces, tener más datos es en realidad tener más ruido. RFE funciona como un **escultor frente a un bloque de mármol**: no busca qué añadir, sino qué sobra. En lugar de probar todas las combinaciones posibles, lo cual sería eterno, RFE entrena un modelo, evalúa la importancia de cada variable y elimina la menos relevante de forma iterativa.

### El Escenario de Elena
Imagina a Elena, una analista de crédito en un banco digital. Ella maneja doscientas variables por cliente, desde ingresos anuales hasta datos triviales. Este exceso de información genera predicciones erráticas. Al aplicar RFE, Elena puede reducir esas doscientas variables a solo las doce esenciales, logrando un modelo más rápido, preciso y fácil de explicar.

**Objetivo:** Demostrar el proceso de selección iterativa mediante RFE de Scikit-Learn en Python para identificar el subconjunto óptimo de variables.

In [ ]:
# Preparación del entorno y creación de datos sintéticos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.feature_selection import RFE, RFECV
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Simulamos el escenario de Elena: 30 variables, pero solo 5 son realmente importantes
X, y = make_regression(n_samples=1000, n_features=30, n_informative=5, noise=0.1, random_state=42)

# Convertimos a DataFrame para visualización
nombres_columnas = [f'Variable_{i}' for i in range(30)]
df_X = pd.DataFrame(X, columns=nombres_columnas)

print('Dataset creado con éxito')
print(f'Forma del dataset: {df_X.shape}')
print()
print('Primeras 5 filas de las primeras 5 columnas:')
print(df_X.head())

### Implementación de RFE

Como mencionamos en el video, el corazón de este proceso es el **Estimador**. Necesitamos un algoritmo que asigne pesos o importancia a las características, como los coeficientes de una regresión.

En este ejemplo, configuraremos RFE para que seleccione específicamente las **5 mejores características** usando una Regresión Lineal como base.

In [ ]:
# 1. Instanciar el estimador base
modelo_base = LinearRegression()

# 2. Configurar RFE para seleccionar 5 variables
selector_rfe = RFE(estimator=modelo_base, n_features_to_select=5, step=1)

# 3. Ajustar el selector a los datos
selector_rfe = selector_rfe.fit(df_X, y)

print('Proceso de eliminación completado.')
print()

### Entendiendo el Ranking

Si llamamos al atributo `ranking_` del objeto RFE, obtendremos una serie de números enteros:
- El número **1** representa a nuestras ganadoras (las seleccionadas).
- Los números superiores (**2, 3, 4...**) nos dicen en qué orden fueron descartadas las demás.

¡Esto nos da un control total y elimina las adivinanzas!

In [ ]:
# Crear un resumen de los resultados
resumen_rfe = pd.DataFrame({
    'Característica': nombres_columnas,
    'Seleccionada': selector_rfe.support_,
    'Ranking': selector_rfe.ranking_
})

# Mostrar las seleccionadas ordenadas por ranking
print('Ranking de características (Top 10):')
print(resumen_rfe.sort_values(by='Ranking').head(10))

### El Nivel Pro: RFECV (Validación Cruzada)

Un error común es fijar el número de características a ciegas. Si pides 5 pero la información vital está en 10, la precisión caerá.

**RFECV** soluciona esto combinando la eliminación recursiva con la **validación cruzada**. Prueba diferentes tamaños de subconjuntos y encuentra automáticamente el punto exacto donde el desempeño es máximo antes de empezar a decaer.

In [ ]:
# Configurar RFECV para encontrar el número óptimo de variables
# Usamos 5 particiones (cv=5) para la validación
selector_rfecv = RFECV(estimator=modelo_base, step=1, cv=5, scoring='neg_mean_squared_error')
selector_rfecv.fit(df_X, y)

print(f'Número óptimo de características sugerido: {selector_rfecv.n_features_}')
print()

# Visualización de la curva de desempeño
plt.figure(figsize=(10, 6))
plt.xlabel('Número de características seleccionadas')
plt.ylabel('Puntaje de Validación Cruzada (Neg MSE)')
plt.title('RFECV: Desempeño vs Cantidad de Variables')
plt.plot(range(1, len(selector_rfecv.cv_results_['mean_test_score']) + 1),
         selector_rfecv.cv_results_['mean_test_score'], marker='o', color='teal')
plt.grid(True)
plt.show()

### Conclusión y Cierre

¡Felicidades! Has dominado una de las técnicas más sofisticadas de selección de características. Con **RFE** y **RFECV** en tu caja de herramientas, estás listo para enfrentar cualquier volumen de datos con total confianza y precisión.

Ya no estamos adivinando qué columnas borrar; ahora tomamos decisiones basadas en la importancia matemática que el propio algoritmo nos revela en cada iteración.

**¡Sigue practicando y nos vemos en la próxima lección!**